In [1]:
import numpy as np
import pandas as pd


In [2]:
movies=pd.read_csv('tmdb_5000_movies.csv')
credits=pd.read_csv('tmdb_5000_credits.csv')    


In [3]:
movies=movies.merge(credits,on='title')

In [4]:
#genere
#id
#keywords
#title
#overview
#cast
#crew
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]






In [5]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [6]:
movies.dropna(inplace=True)

In [7]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [8]:
import ast

In [9]:
def convert(obj):
   l=[]
   for i in  ast.literal_eval(obj):
      l.append(i['name'])
      return l
      
      

In [10]:
movies['genres']=movies['genres'].apply(convert)

In [11]:
movies['keywords']=movies['keywords'].apply(convert)

In [12]:
def convert3(obj):
   l=[]
   counter=0
   for i in  ast.literal_eval(obj):
      if counter!=3:
         l.append(i['name'])
         counter+=1
      else:
         break
   return l      
     

In [13]:
movies['cast']=movies['cast'].apply(convert3)

In [14]:
def fetch_director(obj):
   l=[]
   for i in  ast.literal_eval(obj):
      if i['job']=='Director':
        l.append(i['name'])
        break
   return l

In [15]:
movies['crew']=movies['crew'].apply(fetch_director)

In [16]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",[Action],[culture clash],"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",[Adventure],[ocean],"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,[Action],[spy],"[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,[Action],[dc comics],"[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...",[Action],[based on novel],"[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [17]:
movies['overview'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [18]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [19]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x] if x is not None and isinstance(x, list) else [])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x] if x is not None and isinstance(x, list) else [])
movies['cast'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x] if x is not None and isinstance(x, list) else [])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x] if x is not None and isinstance(x, list) else [])

In [20]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",[Action],[cultureclash],[cultureclash],[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",[Adventure],[ocean],[ocean],[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",[Action],[spy],[spy],[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",[Action],[dccomics],[dccomics],[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",[Action],[basedonnovel],[basedonnovel],[AndrewStanton]


In [21]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [22]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",[Action],[cultureclash],[cultureclash],[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",[Adventure],[ocean],[ocean],[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",[Action],[spy],[spy],[SamMendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",[Action],[dccomics],[dccomics],[ChristopherNolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",[Action],[basedonnovel],[basedonnovel],[AndrewStanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [23]:
new_df=movies[['movie_id','title','tags']]

In [24]:
new_df['tags']=new_df['tags'].apply(lambda x:" ".join(x))

In [25]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [26]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [27]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)  
new_df['tags']=new_df['tags'].apply(stem)

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')

In [29]:
vectors=cv.fit_transform(new_df['tags']).toarray()

In [30]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(vectors)

In [32]:
def recommend(movie):
    movie_index=new_df[new_df['title']==movie].index[0]
    distances=similarity[movie_index]
    movies_list=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
       

In [33]:
recommend("Batman Begins")

Synecdoche, New York
Copying Beethoven
The Animal
The Truman Show
Bruce Almighty


In [34]:
import pickle
pickle.dump(new_df.to_dict(),open('movie_dict.pkl','wb'))

In [35]:
pickle.dump(similarity,open('similarity.pkl','wb'))